In [1]:
import torch
import cv2
import os
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from transformers import BlipProcessor, BlipForConditionalGeneration

MODEL_DIR = os.path.abspath("saved_models_v3")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FONT_PATH = "arial.ttf"

processor = BlipProcessor.from_pretrained(MODEL_DIR)
model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

def predict_caption(image_rgb, max_length=50, num_beams=3):
    inputs = processor(images=image_rgb, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values=inputs["pixel_values"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )
    caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return caption

def put_viet_text(img_bgr, text, pos, font_path, font_size=20, color=(255,255,255), max_width=None):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()
        print("Không tìm thấy font.")

    if max_width is None:
        max_width = pil_img.width - 20

    lines = []
    words = text.split()
    while words:
        line = words.pop(0)
        while words:
            test_line = line + " " + words[0]
            bbox = draw.textbbox((0,0), test_line, font=font)
            if bbox[2] - bbox[0] <= max_width:
                line = test_line
                words.pop(0)
            else:
                break
        lines.append(line)

    x, y = pos
    for line in lines:
        bbox = draw.textbbox((0,0), line, font=font)
        line_h = bbox[3] - bbox[1]
        draw.text((x, y), line, font=font, fill=color, stroke_width=1, stroke_fill=(0,0,0))
        y += line_h + 5

    result_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    return result_bgr

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Không mở được webcam.")
    exit()

caption = None
while True:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()

    if caption:
        display = put_viet_text(display, caption, (10, 30), FONT_PATH, font_size=24, max_width=display.shape[1]-40)

    cv2.imshow("BLIP Vietnamese Captioning", display)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break
    elif key == ord(' '):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        caption = predict_caption(rgb_frame)

cap.release()
cv2.destroyAllWindows()

c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 1866.99it/s]
